In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

In [ ]:
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')

# Data Loading and Inspection

In [ ]:
df_gym= pd.read_csv('data/1. Gym Members Exercise Dataset/gym_members_exercise_tracking.csv')
df_fit = pd.read_csv('data/2. Exercise and Fitness Metrics Dataset/exercise_dataset.csv')
df_excise= pd.read_csv('data/3. Fitness Exercises Dataset/exercises_flat.csv')
df_rec = pd.read_excel('data/4. Mendeley Gym Recommendation Dataset/gym recommendation.xlsx')

print("Columns in gym_members_exercise_tracking.csv:", df_gym.columns.tolist())
print("Columns in exercise_dataset.csv:", df_fit.columns.tolist())
print("Columns in exercises_flat.csv:", df_excise.columns.tolist())
print("Columns in gym recommendation.xlsx:", df_rec.columns.tolist())
print()
print(f'Dataset 1 (Gym Members)          : {df_gym.shape[0]:,} rows × {df_gym.shape[1]} cols')
print(f'Dataset 2 (Exercise & Fitness)   : {df_fit.shape[0]:,} rows × {df_fit.shape[1]} cols')
print(f'Dataset 3 (Exercise Library)     : {df_excise.shape[0]:,} rows × {df_excise.shape[1]} cols')
print(f'Dataset 4 (Mendeley Coaching)    : {df_rec.shape[0]:,} rows × {df_rec.shape[1]} cols')
print()

In [ ]:
print(df_gym.head(3).to_string())
print(df_fit.head(3).to_string())
print(df_excise.head(3).to_string())
print(df_rec.head(3).to_string())


In [ ]:
for label, df in [("D1 Gym Members", df_gym), ("D2 Fitness", df_fit),("D3 Exercises", df_excise), ("D4 Recommendation", df_rec)]:
    missing = df.isnull().sum()
    has_missing = missing[missing > 0]
    if len(has_missing):
        print(f"\n{label}:\n{has_missing}")
    else:
        print(f"{label}: No missing values")

In [ ]:
print("\nDataset 1 (Gym Members) Descriptive Stats")
print(df_gym.describe().round(2).to_string())
 
print("\nDataset 2 (Fitness) Descriptive Stats")
print(df_fit.describe().round(2).to_string())

print("\nDataset 3 (Exercises) Descriptive Stats")
print(df_excise.describe().round(2).to_string())

print("\nDataset 4 (Recommendation) Descriptive Stats")
print(df_rec.describe().round(2).to_string())

In [ ]:
print("Dataset 1 (Gym Members) Info:")
df_gym.info()

print("\nDataset 2 (Fitness) Info:")
df_fit.info()

print("\nDataset 3 (Exercises) Info:")
df_excise.info()

print("\nDataset 4 (Recommendation) Info:")
df_rec.info()

In [ ]:
df_gym.head(3)

In [ ]:
df_gym.head(3)

In [ ]:
df_excise.head(3)

In [ ]:
df_rec.head(3)

# Data Preprocessing

In [ ]:
df_fit_clean = df_fit.copy()

df_fit_clean.drop(columns=['ID', 'Exercise'], inplace=True)

df_fit_clean.rename(columns={
    'Calories Burn' : 'Calories_Burned',
    'Actual Weight' : 'Weight (kg)',
    'Dream Weight' : 'Dream_Weight',
    'Heart Rate' : 'Avg_BPM',
    'Weather Conditions' : 'Weather_Conditions',
    'Exercise Intensity' : 'Exercise_Intensity',
}, inplace=True)

df_fit_clean.rename(columns={'Duration': 'Session_Duration (hours)'}, inplace=True)
df_fit_clean['Session_Duration (hours)'] = df_fit_clean['Session_Duration (hours)'] / 60

#Adding new column name _source to dataset 1 and 2 to identify the source of data
df_gym['_source']      = 'ds1'
df_fit_clean['_source'] = 'ds2'

In [ ]:
print('DS1 columns:', list(df_gym.columns))
print()
print('DS2 columns After CLeaning and Renaming:', list(df_fit_clean.columns))

In [ ]:
# pd.concat fills missing columns with NaN automatically
dfUnified = pd.concat([df_gym, df_fit_clean], ignore_index=True)

print(f'Merged DataFrame shape: {dfUnified.shape}')
print(f'  DS1 (gym data) rows: {(dfUnified["_source"]=="ds1").sum()}')
print(f'  DS2 (fit data) rows: {(dfUnified["_source"]=="ds2").sum()}')
print()
print('Missing values per column:')
print(dfUnified.isnull().sum().to_string())

In [ ]:
dfUnified.head(3)

In [ ]:
numeric_cols = dfUnified.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'Calories_Burned']

for col in numeric_cols:
    if dfUnified[col].isnull().sum() > 0:
        median_val = dfUnified[col].median()
        dfUnified[col].fillna(median_val, inplace=True)
        print(f'  Filled "{col}" NaN with median={median_val:.2f}')
print(dfUnified.isnull().sum().to_string())

In [ ]:
for col in ['Workout_Type', 'Weather_Conditions']:
    if col in dfUnified.columns and dfUnified[col].isnull().sum() > 0:
        mode_val = dfUnified[col].mode()[0]
        dfUnified[col].fillna(mode_val, inplace=True)
        print(f'  Filled "{col}" NaNs with mode="{mode_val}"')

print(dfUnified.isnull().sum().to_string())

In [ ]:
dfUnified.drop(columns=['_source'], inplace=True)
print('Removed "_source" column. Current columns:', list(dfUnified.columns))

print(f'\nRechecking NaNs: {dfUnified.isnull().sum().sum()}')

In [ ]:
beforeLength = len(dfUnified)
dfUnified.drop_duplicates(inplace=True)
print('Duplicates removed:', beforeLength - len(dfUnified))

In [ ]:
Q1 = dfUnified['Calories_Burned'].quantile(0.25)
Q3 = dfUnified['Calories_Burned'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR

In [ ]:
afterLength = len(dfUnified)
dfUnified = dfUnified[(dfUnified['Calories_Burned'] >= lower) & (dfUnified['Calories_Burned'] <= upper)]
print(f'Outliers removed: {afterLength - len(dfUnified)}')

In [ ]:
print('Final Shape:', dfUnified.shape)

In [ ]:
dfUnified['Gender'] = dfUnified['Gender'].map({'Male': 1, 'Female': 0})

dfUnified = pd.get_dummies(dfUnified, columns=['Workout_Type'], prefix='wt', drop_first=False)

weather_map = {'Sunny': 2, 'Cloudy': 1, 'Rainy': 0}
dfUnified['Weather_Conditions'] = dfUnified['Weather_Conditions'].map(weather_map)
dfUnified['Weather_Conditions'].fillna(dfUnified['Weather_Conditions'].median(), inplace=True)

# FIX: was truncated as 'dfUnified[bool_cols] = df' — now correct
bool_cols = dfUnified.select_dtypes(include='bool').columns
dfUnified[bool_cols] = dfUnified[bool_cols].astype(int)

print('Cell 25 fixed: bool columns cast to int correctly.')

In [ ]:
print('Encoded columns:', [c for c in dfUnified.columns if c.startswith('wt_') or c in ['Gender','Weather_Conditions']])
print(f'DataFrame shape after encoding: {dfUnified.shape}')
dfUnified.head(3)

In [ ]:
dfUnified.columns.tolist()

In [ ]:
dfUnified.dtypes

---
## Feature Engineering & Selection (fixed)

In [ ]:
# ── FIX: Root Cause 1 — use only COMMON features (no 82%-imputed columns)
# DS1-only columns that are median-filled for all 3,864 DS2 rows:
#   Height, Max_BPM, Resting_BPM, Fat_Percentage,
#   Water_Intake, Workout_Frequency, Experience_Level
# Using them pollutes the model — they add noise, not signal.

# FIX: Root Cause 2 — add engineered features from COMMON columns only
# HR × Duration: single strongest interaction term
dfUnified['HR_Duration'] = (
    dfUnified['Avg_BPM'] * dfUnified['Session_Duration (hours)']
)
# Workout intensity × time
dfUnified['Intensity_Duration'] = (
    dfUnified['Exercise_Intensity'] * dfUnified['Session_Duration (hours)']
)

COMMON_FEATURES = [
    'Age', 'Gender', 'Weight (kg)', 'BMI',
    'Avg_BPM', 'Session_Duration (hours)',
    'Exercise_Intensity', 'Weather_Conditions',
    'wt_Cardio', 'wt_HIIT', 'wt_Strength', 'wt_Yoga',
    'HR_Duration',         # new: HR × session time
    'Intensity_Duration',  # new: intensity × session time
]

print('Engineered features added.')
print(f'Common feature set size: {len(COMMON_FEATURES)}')
print('Features:', COMMON_FEATURES)

# EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(dfUnified['Calories_Burned'], bins=50, color='navy', edgecolor='white')
axes[0].set_title('Distribution of Calories Burned')
axes[0].set_xlabel('Calories Burned')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(dfUnified['Calories_Burned'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='teal', alpha=0.6))
axes[1].set_title('Calories Burned Boxplot')
axes[1].set_ylabel('Calories Burned')

plt.tight_layout()
plt.show()

In [ ]:
numeric_Df = dfUnified.select_dtypes(include='number')
corr = numeric_Df.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='mako',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
scatter_features = [
    'Session_Duration (hours)', 'Avg_BPM', 'Weight (kg)', 'BMI', 'Age'
]

fig, axes = plt.subplots(1, len(scatter_features), figsize=(18, 4))
for ax, feat in zip(axes, scatter_features):
    ax.scatter(dfUnified[feat], dfUnified['Calories_Burned'],
               alpha=0.2, s=10, color='navy')
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('Calories Burned', fontsize=9)
    ax.set_title(feat, fontsize=9)

plt.suptitle('Feature vs Calories Burned', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
hist_features = ['Age', 'Weight (kg)', 'Height (m)', 'BMI',
                 'Session_Duration (hours)', 'Workout_Frequency (days/week)', 'Water_Intake (liters)', 'Fat_Percentage']
hist_features = [f for f in hist_features if f in dfUnified.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, feat in enumerate(hist_features):
    axes[i].hist(dfUnified[feat].dropna(), bins=30,
                 color=sns.color_palette('muted')[i % 8], edgecolor='white')
    axes[i].set_title(feat, fontsize=9)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gender_means = dfUnified.groupby('Gender')['Calories_Burned'].mean()
axes[0].bar(['Female (0)', 'Male (1)'], gender_means.values,
            color=['pink','navy'])
axes[0].set_title('Average Calories Burned by Gender')
axes[0].set_ylabel('Avg Calories')

wt_cols = [c for c in dfUnified.columns if c.startswith('wt_')]
if wt_cols:
    dfUnified['_wt_label'] = dfUnified[wt_cols].idxmax(axis=1).str.replace('wt_', '')
    wt_means = dfUnified.groupby('_wt_label')['Calories_Burned'].mean().sort_values()
    wt_means.plot(kind='barh', ax=axes[1], color='steelblue')
    axes[1].set_title('Average Calories Burned by Workout Type')
    axes[1].set_xlabel('Avg Calories')
    dfUnified.drop(columns=['_wt_label'], inplace=True)

plt.tight_layout()
plt.show()

# Feature Engineering

In [ ]:
# Feature importance using COMMON_FEATURES only (no imputed-noise cols)
X_all = dfUnified[COMMON_FEATURES].copy()
y_all = dfUnified['Calories_Burned']

rf_quick = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_quick.fit(X_all, y_all)
importance_df = pd.DataFrame({
    'Feature': X_all.columns,
    'ImportanceRF': rf_quick.feature_importances_
})

perm = permutation_importance(
    rf_quick, X_all, y_all,
    n_repeats=10, random_state=42, n_jobs=-1
)
perm_df = pd.DataFrame({
    'Feature': X_all.columns,
    'ImportancePM': perm.importances_mean
})

importance_merged = importance_df.merge(perm_df, on='Feature')
importance_merged['ImportanceRF'] /= importance_merged['ImportanceRF'].max()
importance_merged['ImportancePM'] /= importance_merged['ImportancePM'].max()
importance_merged['FinalScore'] = (
    importance_merged['ImportanceRF'] + importance_merged['ImportancePM']
) / 2
importance_merged = importance_merged.sort_values('FinalScore', ascending=False)
print(importance_merged.to_string(index=False))

# Use all common features (already clean — no threshold needed)
selected_features = COMMON_FEATURES

X = dfUnified[selected_features].copy()
y = dfUnified['Calories_Burned'].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'\nTraining set  : {X_train.shape}')
print(f'Test set      : {X_test.shape}')
print(f'Features used : {len(selected_features)}')

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_merged, x='FinalScore', y='Feature', palette='viridis')
plt.title('Feature Importance (RF + Permutation, common features only)')
plt.tight_layout()
plt.show()

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Scaling done. Train mean:', X_train_sc[:, :3].mean(axis=0).round(3))

# Training Model

In [ ]:
#Helper fn to evaluate models
results = {}

def evaluate(name, model, X_tr, X_te, y_tr, y_te, use_scaled=False):
    Xtr = X_tr if not use_scaled else X_train_sc #train set (scaled or unscaled)
    Xte = X_te if not use_scaled else X_test_sc #test set (scaled or unscaled)
    
    pred = model.predict(Xte)  #predictions on test set
    mae  = mean_absolute_error(y_te, pred) #MAE on test set
    rmse = np.sqrt(mean_squared_error(y_te, pred)) #RMSE on test set
    r2   = r2_score(y_te, pred) #R² on test set
    
    # 5 fold cv R² on full scaled set
    cv_scores = cross_val_score(model, Xtr, y_tr, cv=5,
                                scoring='r2', n_jobs=-1)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2,
                     'CV_R2_mean': cv_scores.mean(), 'CV_R2_std': cv_scores.std()}
    
    print(f'  MAE  = {mae:.2f}')
    print(f'  RMSE = {rmse:.2f}')
    print(f'  R²   = {r2:.4f}')
    print(f'  CV R² = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    return pred

In [ ]:
print('Linear Regression:')
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
pred_lr = evaluate('Linear Regression', lr,
                   X_train_sc, X_test_sc, y_train, y_test,
                   use_scaled=True)

In [ ]:
print('Random Forest Regressor:')
# FIX Root Cause 3: was missing 'rf = ' — object was created but not assigned
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=3,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
pred_rf = evaluate('Random Forest', rf,
                   X_train, X_test, y_train, y_test)

In [ ]:
print('XGBoost Regressor:')
# FIX Root Cause 3: was missing 'xgb = ' — object was created but not assigned
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    verbosity=0
)
xgb.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False)
pred_xgb = evaluate('XGBoost', xgb,
                    X_train, X_test, y_train, y_test)

In [ ]:
print('ANN:')

n_features = X_train_sc.shape[1]

ann = Sequential([
    Dense(128, activation='relu', input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

ann.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='mse',
    metrics=['mae']
)

early_stop = EarlyStopping(
    monitor='val_loss', patience=15,
    restore_best_weights=True
)

lr_sched = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=5, min_lr=1e-5
)

# FIX: was 'y_train_sc' which doesn't exist — target is never scaled
history = ann.fit(
    X_train_sc, y_train,
    validation_split=0.15,
    epochs=300,
    batch_size=32,
    callbacks=[early_stop, lr_sched],
    verbose=0
)

pred_ann = ann.predict(X_test_sc).flatten()
mae_ann  = mean_absolute_error(y_test, pred_ann)
rmse_ann = np.sqrt(mean_squared_error(y_test, pred_ann))
r2_ann   = r2_score(y_test, pred_ann)

results['ANN'] = {'MAE': mae_ann, 'RMSE': rmse_ann, 'R2': r2_ann,
                  'CV_R2_mean': float('nan'), 'CV_R2_std': float('nan')}

print(f'  MAE  = {mae_ann:.2f}')
print(f'  RMSE = {rmse_ann:.2f}')
print(f'  R\u00b2   = {r2_ann:.4f}')
print('  (CV not applicable for Keras models in this setup)')

plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('ANN Training Curve')
plt.legend()
plt.tight_layout()
plt.show()

---
## Model Evaluation

In [ ]:
# ── Model Comparison Table ───────────────────────────────────────────────
results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Model', 'MAE', 'RMSE', 'R\u00b2', 'CV R\u00b2 (mean)', 'CV R\u00b2 (std)']
results_df = results_df.sort_values('R\u00b2', ascending=False).reset_index(drop=True)
results_df[['MAE','RMSE','R\u00b2','CV R\u00b2 (mean)','CV R\u00b2 (std)']] = \
    results_df[['MAE','RMSE','R\u00b2','CV R\u00b2 (mean)','CV R\u00b2 (std)']].round(4)

print('=== Model Comparison ===')
display(results_df)

best_model_name = results_df.iloc[0]['Model']
print(f'\nBest model by R\u00b2: {best_model_name}')

metrics = ['MAE', 'RMSE', 'R\u00b2']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = sns.color_palette('muted', len(results_df))
for ax, metric in zip(axes, metrics):
    bars = ax.bar(results_df['Model'], results_df[metric], color=colors)
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005 * bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
plt.suptitle('Model Comparison')
plt.tight_layout()
plt.show()

In [ ]:
# ── Actual vs Predicted — All 4 Models ──────────────────────────────────
all_preds = {
    'Linear Regression': pred_lr,
    'Random Forest'    : pred_rf,
    'XGBoost'          : pred_xgb,
    'ANN'              : pred_ann,
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for ax, (name, pred) in zip(axes, all_preds.items()):
    ax.scatter(y_test, pred, alpha=0.3, s=10, color='steelblue')
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect fit')
    r2 = results[name]['R2']
    ax.set_title(f'{name}  (R\u00b2={r2:.4f})')
    ax.set_xlabel('Actual Calories')
    ax.set_ylabel('Predicted Calories')
    ax.legend(fontsize=8)
plt.suptitle('Actual vs Predicted — All Models', fontsize=13)
plt.tight_layout()
plt.show()

# Residual analysis on best model
best_pred = all_preds[best_model_name]
residuals = y_test.values - best_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(best_pred, residuals, alpha=0.3, s=10, color='coral')
axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0].set_title(f'Residuals vs Predicted — {best_model_name}')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[1].hist(residuals, bins=40, color='coral', edgecolor='white')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

---
## Ensemble & Save

In [ ]:
# ── Ensemble — average of RF + XGBoost + ANN ────────────────────────────
pred_ensemble = (pred_rf + pred_xgb + pred_ann) / 3

mae_ens  = mean_absolute_error(y_test, pred_ensemble)
rmse_ens = np.sqrt(mean_squared_error(y_test, pred_ensemble))
r2_ens   = r2_score(y_test, pred_ensemble)

results['Ensemble (avg)'] = {
    'MAE': mae_ens, 'RMSE': rmse_ens, 'R2': r2_ens,
    'CV_R2_mean': float('nan'), 'CV_R2_std': float('nan')
}

print('=== Ensemble (RF + XGBoost + ANN average) ===')
print(f'  MAE  = {mae_ens:.2f}')
print(f'  RMSE = {rmse_ens:.2f}')
print(f'  R\u00b2   = {r2_ens:.4f}')

# Update best_model_name if ensemble wins
all_r2 = {name: v['R2'] for name, v in results.items()}
best_model_name = max(all_r2, key=all_r2.get)
print(f'\nOverall best: {best_model_name} (R\u00b2 = {all_r2[best_model_name]:.4f})')

In [ ]:
# ── Save Best Model & Pipeline ───────────────────────────────────────────
import os, json as _json
os.makedirs('out', exist_ok=True)

model_objects = {
    'Linear Regression': lr,
    'Random Forest'    : rf,
    'XGBoost'          : xgb,
}

if best_model_name == 'ANN':
    ann.save('out/best_model_ann.keras')
    print('ANN saved → out/best_model_ann.keras')
elif best_model_name == 'Ensemble (avg)':
    # Save all three component models
    joblib.dump(rf,  'out/ensemble_rf.pkl')
    joblib.dump(xgb, 'out/ensemble_xgb.pkl')
    ann.save('out/ensemble_ann.keras')
    print('Ensemble components saved → out/ensemble_rf.pkl, ensemble_xgb.pkl, ensemble_ann.keras')
else:
    joblib.dump(model_objects[best_model_name], 'out/best_model.pkl')
    print(f'{best_model_name} saved → out/best_model.pkl')

# Always save scaler + feature list + encoding maps
joblib.dump(scaler,            'out/scaler.pkl')
joblib.dump(selected_features, 'out/feature_list.pkl')

wt_cols_saved = [c for c in selected_features if c.startswith('wt_')]
encoding_maps = {
    'weather_map'  : {'Sunny': 2, 'Cloudy': 1, 'Rainy': 0},
    'gender_map'   : {'Male': 1, 'Female': 0},
    'workout_types': [c.replace('wt_', '') for c in wt_cols_saved],
    'feature_list' : selected_features,
}
with open('out/encoding_maps.json', 'w') as f:
    _json.dump(encoding_maps, f, indent=2)

print('\nAll artifacts saved:')
for fname in sorted(os.listdir('out')):
    print(f'  out/{fname}')

In [ ]:
# ── Sanity Check: load model and predict 5 samples ──────────────────────
loaded_scaler   = joblib.load('out/scaler.pkl')
loaded_features = joblib.load('out/feature_list.pkl')

sample_raw = X_test.iloc[:5][loaded_features]
sample_sc  = loaded_scaler.transform(sample_raw)

if best_model_name == 'ANN':
    from tensorflow.keras.models import load_model as _lm
    _m = _lm('out/best_model_ann.keras')
    preds_check = _m.predict(sample_sc).flatten()
elif best_model_name == 'Ensemble (avg)':
    import joblib as _jl
    from tensorflow.keras.models import load_model as _lm
    _rf  = _jl.load('out/ensemble_rf.pkl')
    _xgb = _jl.load('out/ensemble_xgb.pkl')
    _ann = _lm('out/ensemble_ann.keras')
    preds_check = (
        _rf.predict(sample_raw) +
        _xgb.predict(sample_raw) +
        _ann.predict(sample_sc).flatten()
    ) / 3
else:
    _m = joblib.load('out/best_model.pkl')
    if best_model_name == 'Linear Regression':
        preds_check = _m.predict(sample_sc)
    else:
        preds_check = _m.predict(sample_raw)

print(f'Sanity check — {best_model_name} on 5 samples:')
for i, (actual, pred) in enumerate(zip(y_test.values[:5], preds_check)):
    print(f'  [{i+1}] actual={actual:.0f}  predicted={pred:.0f}  diff={abs(actual-pred):.0f}')

print('\nNotebook complete. Model ready for web app integration.')